In [ ]:
import pdfplumber
import pandas as pd
import re

caminho_pdf = 'C:/Users/Z0058B3H/Siemens Energy/Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly/1 - Relatório para extratificação de dados/VP1/E13878.pdf'

with pdfplumber.open(caminho_pdf) as pdf:
    primeira_pag = pdf.pages[0]
    texto_limpo = primeira_pag.extract_text()

print(texto_limpo)

In [ ]:
import pdfplumber
import pandas as pd
import re

caminho_pdf = r"C:\Users\Z0058B3H\Siemens Energy\Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly\1 - Relatório para extratificação de dados\VP1\E13497.pdf"

with pdfplumber.open(caminho_pdf) as pdf:
    primeira_pag = pdf.pages[0]
    texto_limpo = primeira_pag.extract_text()

    
    print("-" * 40)
    print("Iniciando extração da tabela principal...")
    

    todas_as_linhas_tabela = []
    
 
    for num_pagina, pagina in enumerate(pdf.pages):
       
        tabela_extraida = pagina.extract_table()
        print(tabela_extraida)
        
        if tabela_extraida:
            if num_pagina == 0:
                
                todas_as_linhas_tabela.extend(tabela_extraida)
            else:
               
                todas_as_linhas_tabela.extend(tabela_extraida[2:]) 
                

    if todas_as_linhas_tabela:
       
        df_tabela = pd.DataFrame(todas_as_linhas_tabela)
        
   
        caminho_excel_tabela = "C:/Users/Z0058B3H/Downloads/Tabela_Estufa_Completa.xlsx"
        df_tabela.to_excel(caminho_excel_tabela, index=False, header=False)
        
        print(f"Sucesso! Tabela extraída com {len(df_tabela)} linhas.")
        print(f"Salvo em: {caminho_excel_tabela}")
    else:
        print("Nenhuma tabela foi detectada pelo pdfplumber.")

In [ ]:
#Codigo de filtro de VF

import pdfplumber
import pandas as pd
import re

caminho_pdf = r"C:\Users\Z0058B3H\Siemens Energy\Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly\1 - Relatório para extratificação de dados\VP1\E13400.pdf"

with pdfplumber.open(caminho_pdf) as pdf:
    primeira_pag = pdf.pages[0]
    texto_limpo = primeira_pag.extract_text()

    print("-" * 40)
    print("Iniciando extração da tabela principal...")

    todas_as_linhas_tabela = []

    for num_pagina, pagina in enumerate(pdf.pages):
        tabela_extraida = pagina.extract_table()
        # O print abaixo foi comentado para não poluir muito a tela, 
        # mas você pode descomentar se quiser ver os dados puros sendo extraídos.
        # print(tabela_extraida)
        
        if tabela_extraida:
            if num_pagina == 0:
                todas_as_linhas_tabela.extend(tabela_extraida)
            else:
                todas_as_linhas_tabela.extend(tabela_extraida[2:]) 

    if todas_as_linhas_tabela:
        # Cria o DataFrame original
        df_tabela = pd.DataFrame(todas_as_linhas_tabela)
        
        # Troca possíveis campos nulos (None) por texto vazio para evitar erros
        df_tabela = df_tabela.fillna("")

        # ----- INÍCIO DO FILTRO -----
        # Se o pdfplumber separou as colunas direitinho (tem mais de 1 coluna)
        if len(df_tabela.columns) > 1:
            # Filtra para manter apenas as linhas onde a coluna 1 é igual a "VF"
            df_tabela = df_tabela[df_tabela[1].astype(str).str.strip() == "VF"]
        else:
            # Se o pdfplumber colocou a linha inteira na primeira coluna (índice 0)
            # Ele busca linhas que contenham a palavra "VF" isolada
            df_tabela = df_tabela[df_tabela[0].astype(str).str.contains(r'\bVF\b', regex=True)]
        # ----- FIM DO FILTRO -----
        
        caminho_excel_tabela = "C:/Users/Z0058B3H/Downloads/Tabela_Estufa_Completa.xlsx"
        df_tabela.to_excel(caminho_excel_tabela, index=False, header=False)
        
        print(f"Sucesso! Tabela extraída e filtrada com {len(df_tabela)} linhas da fase VF.")
        print(f"Salvo em: {caminho_excel_tabela}")
    else:
        print("Nenhuma tabela foi detectada pelo pdfplumber.")

In [ ]:
#limpa colunas inuteis
import pdfplumber
import pandas as pd
import re

caminho_pdf = r"C:\Users\Z0058B3H\Siemens Energy\Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly\1 - Relatório para extratificação de dados\VP1\E13400.pdf"

with pdfplumber.open(caminho_pdf) as pdf:
    primeira_pag = pdf.pages[0]
    texto_limpo = primeira_pag.extract_text()

    print("-" * 40)
    print("Iniciando extração da tabela principal...")

    todas_as_linhas_tabela = []

    for num_pagina, pagina in enumerate(pdf.pages):
        tabela_extraida = pagina.extract_table()
        
        if tabela_extraida:
            if num_pagina == 0:
                todas_as_linhas_tabela.extend(tabela_extraida)
            else:
                todas_as_linhas_tabela.extend(tabela_extraida[2:]) 

    if todas_as_linhas_tabela:
        # Cria o DataFrame original
        df_tabela = pd.DataFrame(todas_as_linhas_tabela)
        
        # Troca possíveis campos nulos (None) por texto vazio para evitar erros
        df_tabela = df_tabela.fillna("")

        # ----- INÍCIO DO FILTRO DE FASE E REMOÇÃO DE COLUNAS -----
        # Se o pdfplumber separou as colunas direitinho
        if len(df_tabela.columns) > 1:
            # 1. Filtra para manter apenas a fase "VF"
            df_tabela = df_tabela[df_tabela[1].astype(str).str.strip() == "VF"]
            
            # 2. Remove as 6 colunas de Condensado de Água antes do Operador
            # O índice -1 é o Operador. Queremos remover do -7 ao -2.
            colunas_para_remover = df_tabela.columns[-7:-1]
            df_tabela = df_tabela.drop(columns=colunas_para_remover)
            
        else:
            # Se o pdfplumber colocou a linha inteira em uma única coluna
            # 1. Filtra a palavra "VF"
            df_tabela = df_tabela[df_tabela[0].astype(str).str.contains(r'\bVF\b', regex=True)]
            
            # 2. Usa Expressão Regular para apagar os 6 valores numéricos antes do nome do Operador
            df_tabela[0] = df_tabela[0].str.replace(r'(\s+[\d,]+){6}\s+([a-zA-Z]+)$', r' \2', regex=True)
        # ----- FIM DO FILTRO E REMOÇÃO -----
        
        caminho_excel_tabela = "C:/Users/Z0058B3H/Downloads/Tabela_Estufa_Completa.xlsx"
        df_tabela.to_excel(caminho_excel_tabela, index=False, header=False)
        
        print(f"Sucesso! Tabela extraída, filtrada e colunas removidas. Restam {len(df_tabela)} linhas.")
        print(f"Salvo em: {caminho_excel_tabela}")
    else:
        print("Nenhuma tabela foi detectada pelo pdfplumber.")

In [ ]:
import pdfplumber
import pandas as pd
import re

caminho_pdf = r"C:\Users\Z0058B3H\Siemens Energy\Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly\1 - Relatório para extratificação de dados\VP1\E13497.pdf"

with pdfplumber.open(caminho_pdf) as pdf:
    primeira_pag = pdf.pages[0]
    texto_limpo = primeira_pag.extract_text()

    print("-" * 40)
    print("Iniciando extração da tabela principal...")

    todas_as_linhas_tabela = []

    for num_pagina, pagina in enumerate(pdf.pages):
        tabela_extraida = pagina.extract_table()
        
        if tabela_extraida:
            if num_pagina == 0:
                todas_as_linhas_tabela.extend(tabela_extraida)
            else:
                todas_as_linhas_tabela.extend(tabela_extraida[2:]) 

    if todas_as_linhas_tabela:
        # Cria o DataFrame original
        df_tabela = pd.DataFrame(todas_as_linhas_tabela)
        
        # Troca possíveis campos nulos (None) por texto vazio para evitar erros
        df_tabela = df_tabela.fillna("")

        # ----- INÍCIO DO FILTRO DE FASE E REMOÇÃO DE COLUNAS -----
        # Se o pdfplumber separou as colunas direitinho
        if len(df_tabela.columns) > 1:
            # 1. Filtra para manter apenas a fase "VF"
            df_tabela = df_tabela[df_tabela[1].astype(str).str.strip() == "VF"]
            
            # 2. Remove as últimas 7 colunas (6 de condensado de água + 1 do Operador)
            # O índice -7: significa "da sétima coluna de trás para frente até a última"
            colunas_para_remover = df_tabela.columns[-7:]
            df_tabela = df_tabela.drop(columns=colunas_para_remover)
            
        else:
            # Se o pdfplumber colocou a linha inteira em uma única coluna
            # 1. Filtra a palavra "VF"
            df_tabela = df_tabela[df_tabela[0].astype(str).str.contains(r'\bVF\b', regex=True)]
            
            # 2. Usa Expressão Regular para apagar os 6 valores numéricos E o nome do Operador no final da linha
            df_tabela[0] = df_tabela[0].str.replace(r'(\s+[\d,]+){6}\s+\S+$', '', regex=True)
        # ----- FIM DO FILTRO E REMOÇÃO -----
        
        caminho_excel_tabela = "C:/Users/Z0058B3H/Downloads/Tabela_Estufa_Completa.xlsx"
        df_tabela.to_excel(caminho_excel_tabela, index=False, header=False)
        
        print(f"Sucesso! Tabela extraída, filtrada e colunas finais removidas. Restam {len(df_tabela)} linhas.")
        print(f"Salvo em: {caminho_excel_tabela}")
    else:
        print("Nenhuma tabela foi detectada pelo pdfplumber.")

In [ ]:
#teste cabeçalho
import pdfplumber
import pandas as pd
import re

caminho_pdf = r"C:\Users\Z0058B3H\Siemens Energy\Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly\1 - Relatório para extratificação de dados\VP1\E13497.pdf"

with pdfplumber.open(caminho_pdf) as pdf:
    primeira_pag = pdf.pages[0]
    texto_limpo = primeira_pag.extract_text()

    print("-" * 40)
    print("Iniciando extração da tabela principal...")

    todas_as_linhas_tabela = []

    for num_pagina, pagina in enumerate(pdf.pages):
        tabela_extraida = pagina.extract_table()
        
        if tabela_extraida:
            if num_pagina == 0:
                todas_as_linhas_tabela.extend(tabela_extraida)
            else:
                # Pula as duas primeiras linhas nas páginas seguintes (cabeçalhos repetidos)
                todas_as_linhas_tabela.extend(tabela_extraida[2:]) 

    if todas_as_linhas_tabela:
        # Cria o DataFrame original
        df_tabela = pd.DataFrame(todas_as_linhas_tabela)
        
        # Troca possíveis campos nulos (None) por texto vazio para evitar erros
        df_tabela = df_tabela.fillna("")

        # ----- INÍCIO DO FILTRO DE FASE E REMOÇÃO DE COLUNAS -----
        # Se o pdfplumber separou as colunas direitinho (mais de 1 coluna)
        if len(df_tabela.columns) > 1:
            # 1. Filtra para manter apenas a fase "VF"
            df_tabela = df_tabela[df_tabela[1].astype(str).str.strip() == "VF"]
            
            # 2. Remove as últimas 7 colunas (6 de condensado de água + 1 do Operador)
            colunas_para_remover = df_tabela.columns[-7:]
            df_tabela = df_tabela.drop(columns=colunas_para_remover)
            
            # 3. NOMEANDO AS COLUNAS
            # Baseado na sua imagem, sobraram 18 colunas de dados úteis.
            titulos_colunas = [
                "Dia-hora", "Fase", "Setp (Evaporador)", "Saida (Evaporador)", 
                "Ret (Evaporador)", "Evap (Evaporador)", "Evap Vác. (mbar)", 
                "T1 (Paredes)", "T2 (Paredes)", "T3 (Paredes)", "T4 (Paredes)", 
                "Sup (Jugo)", "Inf (Jugo)", "Sup (Bobina)", "Inf (Bobina)", 
                "Pirani (Vácuo mbar)", "Baratron (Vácuo mbar)", "Extr. Umid. (g/h.ton)"
            ]
            
            # Garante que o número de títulos bate com o número de colunas
            if len(df_tabela.columns) == len(titulos_colunas):
                df_tabela.columns = titulos_colunas
            else:
                print(f"Aviso: O número de colunas ({len(df_tabela.columns)}) é diferente do número de títulos ({len(titulos_colunas)}).")

        else:
            # Se o pdfplumber colocou a linha inteira em uma única coluna
            df_tabela = df_tabela[df_tabela[0].astype(str).str.contains(r'\bVF\b', regex=True)]
            df_tabela[0] = df_tabela[0].str.replace(r'(\s+[\d,]+){6}\s+\S+$', '', regex=True)
            
            # Para texto único, precisaremos dividir a string em colunas reais primeiro
            # Isso divide a linha única em múltiplas colunas separadas por espaço
            df_tabela = df_tabela[0].str.split(expand=True)
            
            titulos_colunas = [
                "Dia-hora", "Fase", "Setp", "Saida", "Ret", "Evap", "Evap_mbar", 
                "T1", "T2", "T3", "T4", "Sup_Jugo", "Inf_Jugo", "Sup_Bobina", 
                "Inf_Bobina", "Pirani", "Baratron", "Extr_Umid"
            ]
            
            if len(df_tabela.columns) == len(titulos_colunas):
                df_tabela.columns = titulos_colunas
        # ----- FIM DO FILTRO E REMOÇÃO -----
        
        caminho_excel_tabela = "C:/Users/Z0058B3H/Downloads/Tabela_Estufa_Completa.xlsx"
        s
        # IMPORTANTE: header=True para que os títulos apareçam no Excel
        df_tabela.to_excel(caminho_excel_tabela, index=False, header=True)
        
        print(f"Sucesso! Tabela extraída, filtrada e colunas finais removidas. Restam {len(df_tabela)} linhas.")
        print(f"Salvo em: {caminho_excel_tabela}")
    else:
        print("Nenhuma tabela foi detectada pelo pdfplumber.")